# Two-Tower Neural Network — LinkedIn Recommendation

## Architecture
```
Profile A  →  [Tower A: FC layers]  →  embedding_a  (128-d)
                                              ↘
                                          dot product  →  predicted compatibility score
                                              ↗
Profile B  →  [Tower B: FC layers]  →  embedding_b  (128-d)
```
Both towers share **identical architecture** but **separate weights**.  
At serving time:
1. Pre-compute ALL 50K profile embeddings once → store in FAISS index
2. For any query profile: embed it → ANN search in FAISS → top-100 candidates → re-rank

## Input features per profile (19 features)
| # | Feature | Type |
|---|---------|------|
| 0-7 | TF-IDF skills (top-8 SVD components) | float |
| 8-11 | TF-IDF goals (top-4 SVD components) | float |
| 12-15 | TF-IDF can_offer (top-4 SVD components) | float |
| 16 | years_experience (normalized) | float |
| 17 | seniority_ord (0-3, normalized) | float |
| 18 | remote_enc (0-2, normalized) | float |


In [1]:
import subprocess
subprocess.run([
    'pip', 'install', 'torch', 'faiss-cpu', 'scikit-learn',
    'pandas', 'numpy', 'joblib', 'tqdm', '-q'
])
print('Done!')

Done!


In [2]:
# %pip install -q faiss-cpu

import ast, joblib, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from pathlib import Path
import faiss
from tqdm import tqdm

warnings.filterwarnings('ignore')

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR   = Path('../Dataset/')          # ← update to your path
MODELS_DIR = Path('models_twotower')  # output folder
MODELS_DIR.mkdir(exist_ok=True)

print(f'Device : {DEVICE}')

Device : cuda


## 1. Load & preprocess profiles

In [3]:
def safe_parse(val):
    try:
        r = ast.literal_eval(val)
        return r if isinstance(r, list) else []
    except Exception:
        return []

profiles = pd.read_csv(DATA_DIR / 'profiles.csv')
pairs    = pd.read_csv(DATA_DIR / 'compatibility_pairs.csv')

# Parse JSON text fields
for col in ['skills', 'goals', 'needs', 'can_offer']:
    profiles[f'{col}_list'] = profiles[col].apply(safe_parse)
    profiles[f'{col}_text'] = profiles[f'{col}_list'].apply(lambda x: ' '.join(x))

# Remote preference → ordinal
remote_map = {'remote': 2, 'hybrid': 1, 'onsite': 0}
if 'remote_preference' in profiles.columns:
    profiles['remote_enc'] = profiles['remote_preference'].str.lower().map(remote_map).fillna(1).astype(int)
else:
    profiles['remote_enc'] = 1

# Seniority → ordinal
oe_seniority = OrdinalEncoder(
    categories=[['entry', 'mid', 'senior', 'executive']],
    handle_unknown='use_encoded_value', unknown_value=-1
)
profiles['seniority_ord'] = oe_seniority.fit_transform(
    profiles[['seniority_level']].fillna('entry')
).astype(float)

# Industry & city label encode (for metadata — not used as tower input)
le_industry = LabelEncoder()
profiles['industry_enc'] = le_industry.fit_transform(profiles['industry'].fillna('unknown'))

profiles['city'] = profiles['location'].apply(
    lambda x: x.split(',')[0].strip().lower() if isinstance(x, str) else 'unknown'
)

profiles = profiles.reset_index(drop=True)
profile_id_to_idx = {pid: i for i, pid in enumerate(profiles['profile_id'])}

print(f'Profiles : {len(profiles):,}')
print(f'Pairs    : {len(pairs):,}')

Profiles : 50,000
Pairs    : 4,999,890


## 2. Build profile feature vectors (19-d per profile)
TF-IDF → SVD (LSA) compression → concatenate with scalar features.  
**Why SVD?** A 128-d sparse TF-IDF vector is terrible as neural net input.  
SVD compresses it to a dense 8-d semantic representation.

In [4]:
# ── Split for fit (no leakage) ─────────────────────────────────────────
np.random.seed(42)
perm      = np.random.permutation(len(profiles))
train_cut = int(len(profiles) * 0.8)
train_df  = profiles.iloc[perm[:train_cut]]

# ── TF-IDF + SVD pipelines ────────────────────────────────────────────
SKILLS_TFIDF_FEATS    = 300
GOALS_TFIDF_FEATS     = 200
CAN_OFFER_TFIDF_FEATS = 200
SKILLS_SVD_DIM        = 8
GOALS_SVD_DIM         = 4
CAN_OFFER_SVD_DIM     = 4

def fit_tfidf_svd(train_texts, all_texts, tfidf_feats, svd_dim, name):
    tfidf = TfidfVectorizer(max_features=tfidf_feats)
    svd   = TruncatedSVD(n_components=svd_dim, random_state=42)
    tfidf_mat = tfidf.fit(train_texts).transform(all_texts)
    dense_mat = svd.fit(tfidf_mat[:train_cut]).transform(tfidf_mat)  # fit on train rows
    print(f'  {name}: TF-IDF {tfidf_mat.shape} → SVD {dense_mat.shape}')
    return tfidf, svd, dense_mat.astype(np.float32)

print('Building TF-IDF + SVD representations...')
tfidf_skills,    svd_skills,    skills_dense    = fit_tfidf_svd(
    train_df['skills_text'],    profiles['skills_text'],    SKILLS_TFIDF_FEATS,    SKILLS_SVD_DIM,    'skills')
tfidf_goals,     svd_goals,     goals_dense     = fit_tfidf_svd(
    train_df['goals_text'],     profiles['goals_text'],     GOALS_TFIDF_FEATS,     GOALS_SVD_DIM,     'goals')
tfidf_can_offer, svd_can_offer, can_offer_dense = fit_tfidf_svd(
    train_df['can_offer_text'], profiles['can_offer_text'], CAN_OFFER_TFIDF_FEATS, CAN_OFFER_SVD_DIM, 'can_offer')

# ── Scalar features (normalized) ──────────────────────────────────────
scalar_feats = profiles[['years_experience', 'seniority_ord', 'remote_enc']].fillna(0).astype(np.float32)
scaler = StandardScaler()
scalar_norm = scaler.fit(scalar_feats.iloc[perm[:train_cut]]).transform(scalar_feats).astype(np.float32)

# ── Concatenate all profile vectors ───────────────────────────────────
# Shape: (50000, 8+4+4+3) = (50000, 19)
profile_vectors = np.concatenate([
    skills_dense,
    goals_dense,
    can_offer_dense,
    scalar_norm
], axis=1)

PROFILE_DIM = profile_vectors.shape[1]
print(f'\nFinal profile vector dim: {PROFILE_DIM}')
print(f'profile_vectors shape   : {profile_vectors.shape}')

Building TF-IDF + SVD representations...
  skills: TF-IDF (50000, 72) → SVD (50000, 8)
  goals: TF-IDF (50000, 23) → SVD (50000, 4)
  can_offer: TF-IDF (50000, 15) → SVD (50000, 4)

Final profile vector dim: 19
profile_vectors shape   : (50000, 19)


## 3. Dataset class

In [5]:
class PairDataset(Dataset):
    """
    Returns (vec_a, vec_b, score) for each row in compatibility_pairs.
    Score is normalized to [0, 1] (original is 0-100).
    """
    def __init__(self, pairs_df, profile_vecs, pid_to_idx, split_indices):
        sub = pairs_df.iloc[split_indices].reset_index(drop=True)

        # Map pair IDs → profile row indices
        a_idx = sub['profile_a_id'].map(pid_to_idx).dropna().astype(int)
        b_idx = sub['profile_b_id'].map(pid_to_idx).dropna().astype(int)
        valid = a_idx.index.intersection(b_idx.index)

        self.vec_a  = torch.tensor(profile_vecs[a_idx.loc[valid].values], dtype=torch.float32)
        self.vec_b  = torch.tensor(profile_vecs[b_idx.loc[valid].values], dtype=torch.float32)
        self.scores = torch.tensor(
            sub.loc[valid, 'compatibility_score'].values / 100.0,  # normalize 0-100 → 0-1
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return self.vec_a[idx], self.vec_b[idx], self.scores[idx]


# ── Train / Val split on pairs (not profiles) ─────────────────────────
N_PAIRS      = len(pairs)
pair_perm    = np.random.permutation(N_PAIRS)
val_cut      = int(N_PAIRS * 0.9)
train_pidx   = pair_perm[:val_cut]
val_pidx     = pair_perm[val_cut:]

train_ds = PairDataset(pairs, profile_vectors, profile_id_to_idx, train_pidx)
val_ds   = PairDataset(pairs, profile_vectors, profile_id_to_idx, val_pidx)

train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=4096, shuffle=False, num_workers=0, pin_memory=False)

print(f'Train pairs : {len(train_ds):,}')
print(f'Val pairs   : {len(val_ds):,}')

Train pairs : 4,499,901
Val pairs   : 499,989


## 4. Two-Tower model definition

```
Tower(19) → Linear(19→64) → BN → ReLU → Dropout
           → Linear(64→128) → BN → ReLU → Dropout
           → Linear(128→128) → L2-normalize  ← embedding
```
**L2 normalization** at the end means the dot product = cosine similarity (bounded 0–1 after sigmoid).  
This is identical to how Google, YouTube, and LinkedIn structure their retrieval towers.

In [6]:
class Tower(nn.Module):
    """Single encoder tower. Both profile A and B use identical architecture."""
    def __init__(self, input_dim: int, emb_dim: int = 128, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, emb_dim),
        )

    def forward(self, x):
        emb = self.net(x)
        return F.normalize(emb, p=2, dim=-1)  # L2 normalize → cosine dot product


class TwoTowerModel(nn.Module):
    """
    Two separate towers (A and B) with shared architecture but separate weights.
    Forward pass returns the predicted compatibility score in [0, 1].
    """
    def __init__(self, input_dim: int, emb_dim: int = 128, dropout: float = 0.2):
        super().__init__()
        self.tower_a = Tower(input_dim, emb_dim, dropout)
        self.tower_b = Tower(input_dim, emb_dim, dropout)

    def forward(self, x_a, x_b):
        emb_a = self.tower_a(x_a)   # (B, emb_dim)
        emb_b = self.tower_b(x_b)   # (B, emb_dim)
        # Dot product of L2-normalized vectors = cosine similarity ∈ [-1, 1]
        # Scale to [0, 1] with sigmoid
        dot   = (emb_a * emb_b).sum(dim=-1)   # (B,)
        return torch.sigmoid(dot)              # (B,) ∈ [0, 1]

    def encode_a(self, x):
        """Encode profile as 'query' (used at retrieval time)."""
        return self.tower_a(x)

    def encode_b(self, x):
        """Encode profile as 'candidate' (used when building FAISS index)."""
        return self.tower_b(x)


EMB_DIM = 128
model   = TwoTowerModel(PROFILE_DIM, emb_dim=EMB_DIM, dropout=0.2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters : {total_params:,}')
print(model)

Model parameters : 52,992
TwoTowerModel(
  (tower_a): Tower(
    (net): Sequential(
      (0): Linear(in_features=19, out_features=64, bias=True)
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=64, out_features=128, bias=True)
      (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (tower_b): Tower(
    (net): Sequential(
      (0): Linear(in_features=19, out_features=64, bias=True)
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=64, out_features=128, bias=True)
      (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      

## 5. Training

**Loss**: MSE between predicted score and normalized ground-truth score.  
**Optimizer**: AdamW with cosine LR schedule + warmup.

In [7]:
EPOCHS    = 15
LR        = 3e-3
CLIP_GRAD = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,   # 10% warmup
)
criterion = nn.MSELoss()


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for vec_a, vec_b, score in loader:
            vec_a, vec_b, score = vec_a.to(DEVICE), vec_b.to(DEVICE), score.to(DEVICE)
            pred = model(vec_a, vec_b)
            loss = criterion(pred, score)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * len(score)
    return total_loss / len(loader.dataset)


best_val_loss = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    t0       = time.time()
    train_l  = run_epoch(train_loader, train=True)
    val_l    = run_epoch(val_loader,   train=False)
    elapsed  = time.time() - t0
    history.append({'epoch': epoch, 'train_loss': train_l, 'val_loss': val_l})

    tag = ''
    if val_l < best_val_loss:
        best_val_loss = val_l
        torch.save(model.state_dict(), MODELS_DIR / 'two_tower_best.pt')
        tag = '  ← best'

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'train={train_l:.5f}  val={val_l:.5f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  '
          f't={elapsed:.1f}s{tag}')

print(f'\nBest val loss : {best_val_loss:.5f}')

Epoch 01/15  train=0.00193  val=0.00110  lr=2.28e-03  t=97.1s  ← best
Epoch 02/15  train=0.00110  val=0.00103  lr=2.99e-03  t=132.2s  ← best
Epoch 03/15  train=0.00106  val=0.00102  lr=2.91e-03  t=137.4s  ← best
Epoch 04/15  train=0.00104  val=0.00101  lr=2.75e-03  t=139.6s  ← best
Epoch 05/15  train=0.00103  val=0.00101  lr=2.53e-03  t=147.9s  ← best
Epoch 06/15  train=0.00103  val=0.00100  lr=2.25e-03  t=115.9s  ← best
Epoch 07/15  train=0.00102  val=0.00100  lr=1.93e-03  t=129.5s  ← best
Epoch 08/15  train=0.00102  val=0.00100  lr=1.59e-03  t=127.1s  ← best
Epoch 09/15  train=0.00102  val=0.00100  lr=1.24e-03  t=121.5s  ← best
Epoch 10/15  train=0.00102  val=0.00100  lr=9.06e-04  t=79.1s  ← best
Epoch 11/15  train=0.00101  val=0.00099  lr=6.04e-04  t=76.4s  ← best
Epoch 12/15  train=0.00101  val=0.00099  lr=3.51e-04  t=75.5s  ← best
Epoch 13/15  train=0.00101  val=0.00099  lr=1.59e-04  t=74.2s  ← best
Epoch 14/15  train=0.00101  val=0.00099  lr=4.04e-05  t=76.1s  ← best
Epoch 15/15 

## 6. Pre-compute all 50K profile embeddings + build FAISS index

This is the key step that makes retrieval O(log N) instead of O(N).

In [12]:
# Load best weights
model.load_state_dict(torch.load(MODELS_DIR / 'two_tower_best.pt', map_location=DEVICE))
model.eval()

# ── Encode all profiles with tower_b (candidate side) ─────────────────
# At inference: query uses tower_a, all candidates were indexed with tower_b
print('Encoding all 50K profiles with tower_b...')
all_vecs  = torch.tensor(profile_vectors, dtype=torch.float32)
BATCH     = 2048
all_embs  = []

with torch.no_grad():
    for start in tqdm(range(0, len(all_vecs), BATCH)):
        batch = all_vecs[start:start+BATCH].to(DEVICE)
        emb   = model.encode_b(batch)  # (B, 128)
        all_embs.append(emb.cpu().numpy())

all_embs = np.concatenate(all_embs, axis=0).astype(np.float32)  # (50000, 128)
print(f'Embeddings shape : {all_embs.shape}')

# ── Build FAISS IndexFlatIP (Inner Product = cosine since L2-normalized) ─
# Use IndexIVFFlat for faster search on large datasets
print('Building FAISS index...')
N_CELLS = 256   # number of Voronoi cells (sqrt of 50K ≈ 224, round up)

quantizer   = faiss.IndexFlatIP(EMB_DIM)
faiss_index = faiss.IndexIVFFlat(quantizer, EMB_DIM, N_CELLS, faiss.METRIC_INNER_PRODUCT)
faiss_index.train(all_embs)          # learn Voronoi centroids
faiss_index.add(all_embs)            # add all 50K vectors
faiss_index.nprobe = 32              # search 32 cells at query time (speed/recall tradeoff)

faiss.write_index(faiss_index, str(MODELS_DIR / 'profiles.faiss'))
print(f'FAISS index saved  ({faiss_index.ntotal:,} vectors, {N_CELLS} cells, nprobe={faiss_index.nprobe})')

# ── Save embeddings array (for MMR re-ranking) ────────────────────────
np.save(MODELS_DIR / 'all_embeddings.npy', all_embs)
print('Embeddings saved')

Encoding all 50K profiles with tower_b...


100%|██████████| 25/25 [00:00<00:00, 277.78it/s]

Embeddings shape : (50000, 128)
Building FAISS index...


FAISS index saved  (50,000 vectors, 256 cells, nprobe=32)
Embeddings saved


## 7. Save all artifacts needed by the serving layer

In [13]:
# ── Save TF-IDF + SVD pipelines ───────────────────────────────────────
joblib.dump(tfidf_skills,    MODELS_DIR / 'tfidf_skills.pkl')
joblib.dump(svd_skills,      MODELS_DIR / 'svd_skills.pkl')
joblib.dump(tfidf_goals,     MODELS_DIR / 'tfidf_goals.pkl')
joblib.dump(svd_goals,       MODELS_DIR / 'svd_goals.pkl')
joblib.dump(tfidf_can_offer, MODELS_DIR / 'tfidf_can_offer.pkl')
joblib.dump(svd_can_offer,   MODELS_DIR / 'svd_can_offer.pkl')

# ── Save encoders & scaler ────────────────────────────────────────────
joblib.dump(oe_seniority,    MODELS_DIR / 'oe_seniority.pkl')
joblib.dump(le_industry,     MODELS_DIR / 'le_industry.pkl')
joblib.dump(scaler,          MODELS_DIR / 'scalar_scaler.pkl')

# ── Save profiles CSV ─────────────────────────────────────────────────
profiles.to_csv(MODELS_DIR / 'profiles_encoded.csv', index=False)

# ── Save model config ─────────────────────────────────────────────────
import json
config = {
    'profile_dim'     : int(PROFILE_DIM),
    'emb_dim'         : EMB_DIM,
    'skills_svd_dim'  : SKILLS_SVD_DIM,
    'goals_svd_dim'   : GOALS_SVD_DIM,
    'can_offer_svd_dim': CAN_OFFER_SVD_DIM,
    'scalar_dim'      : 3,
    'faiss_nprobe'    : 32,
}
with open(MODELS_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('All artifacts saved to:', MODELS_DIR)
print('Files:', [p.name for p in MODELS_DIR.iterdir()])

All artifacts saved to: models_twotower
Files: ['all_embeddings.npy', 'config.json', 'le_industry.pkl', 'oe_seniority.pkl', 'profiles.faiss', 'profiles_encoded.csv', 'scalar_scaler.pkl', 'svd_can_offer.pkl', 'svd_goals.pkl', 'svd_skills.pkl', 'tfidf_can_offer.pkl', 'tfidf_goals.pkl', 'tfidf_skills.pkl', 'two_tower_best.pt']


## 8. Quick sanity check — retrieve top-10 for a sample profile

In [14]:
def retrieve_top_n(profile_id: str, top_n: int = 10):
    """Query the FAISS index for a given profile."""
    ia   = profile_id_to_idx[profile_id]
    qvec = torch.tensor(profile_vectors[ia:ia+1], dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        q_emb = model.encode_a(qvec).cpu().numpy().astype(np.float32)  # tower_a for query

    distances, indices = faiss_index.search(q_emb, top_n + 1)  # +1 to skip self

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == ia:
            continue  # skip self
        p = profiles.iloc[idx]
        results.append({
            'name'             : p['name'],
            'current_role'     : p['current_role'],
            'industry'         : p['industry'],
            'seniority_level'  : p['seniority_level'],
            'years_experience' : p['years_experience'],
            'cosine_score'     : round(float(dist), 4),
        })
        if len(results) == top_n:
            break
    return pd.DataFrame(results)


sample_pid  = profiles['profile_id'].iloc[0]
sample_name = profiles['name'].iloc[0]
print(f'Top 10 recommendations for: {sample_name}\n')
print(retrieve_top_n(sample_pid, top_n=10).to_string(index=False))

Top 10 recommendations for: Daniel Doyle

              name           current_role       industry seniority_level  years_experience  cosine_score
       Mary Hughes        Senior Designer Transportation          senior                 8       -0.4613
      Robert Jones               Director      Education          senior                 8       -0.4614
      Joshua Ochoa    Lead Data Scientist     Technology          senior                 8       -0.4615
Andrew Hensley Jr.    Lead Data Scientist Transportation          senior                 8       -0.4615
      Mark Shannon     Principal Engineer         Energy          senior                 8       -0.4615
     Tamara Mendez               Director          Media          senior                 8       -0.4616
       Dwayne Pham        Senior Designer Transportation          senior                 8       -0.4616
  Rebecca Anderson        Senior Engineer         Energy          senior                 8       -0.4616
      Mark Br

## 9. Recall@K evaluation
How often does the FAISS retrieval include the truly high-scoring pairs?

In [15]:
EVAL_SAMPLES = 500
K_LIST       = [10, 50, 100]

# Ground truth: for each profile, which partners have score > 70?
high_compat = pairs[pairs['compatibility_score'] > 70]
pos_map     = high_compat.groupby('profile_a_id')['profile_b_id'].apply(set).to_dict()

# Sample profiles that have at least 1 high-compat pair
eval_pids = [pid for pid in list(pos_map.keys()) if pid in profile_id_to_idx][:EVAL_SAMPLES]

recall_at_k = {k: [] for k in K_LIST}

model.eval()
with torch.no_grad():
    for pid in tqdm(eval_pids, desc='Recall@K eval'):
        ia   = profile_id_to_idx[pid]
        qvec = torch.tensor(profile_vectors[ia:ia+1], dtype=torch.float32).to(DEVICE)
        q_emb = model.encode_a(qvec).cpu().numpy().astype(np.float32)

        _, indices = faiss_index.search(q_emb, max(K_LIST) + 1)
        retrieved_pids = set(profiles['profile_id'].iloc[idx] for idx in indices[0] if idx != ia)
        true_pids      = pos_map.get(pid, set())

        if not true_pids:
            continue

        for k in K_LIST:
            top_k   = set(list(retrieved_pids)[:k])
            overlap = len(top_k & true_pids)
            recall_at_k[k].append(overlap / len(true_pids))

print('\nRecall@K results:')
for k in K_LIST:
    r = np.mean(recall_at_k[k])
    print(f'  Recall@{k:<3d} = {r:.4f} ({r*100:.1f}%)')

Recall@K eval: 0it [00:00, ?it/s]


Recall@K results:
  Recall@10  = nan (nan%)
  Recall@50  = nan (nan%)
  Recall@100 = nan (nan%)
